In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

BRONZE_SCHEMA = "supplysage_bronze"
SILVER_SCHEMA = "supplysage_silver"

TRANSFORM_RUN_ID = f"silver_external_risk_events_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "13_silver_external_risk_events"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

external_bronze_tables = [
    "bronze_ext_gdelt_doc_raw",
    "bronze_ext_nws_alerts_raw",
    "bronze_ext_cisa_kev_raw",
    "bronze_ext_cpsc_recalls_raw",
    "bronze_ext_ofac_sanctions_raw",
    "bronze_ext_sec_company_submissions_raw",
    "bronze_ext_eia_fuel_prices_raw",
    "bronze_ext_ingestion_batch_manifest"
]

print("TRANSFORM_RUN_ID:", TRANSFORM_RUN_ID)
print("BRONZE_SCHEMA:", BRONZE_SCHEMA)
print("SILVER_SCHEMA:", SILVER_SCHEMA)
print("NOTEBOOK_NAME:", NOTEBOOK_NAME)

# Use latest clean external ingestion batch from the manifest.
latest_manifest_row = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_ingestion_batch_manifest")
    .filter(F.col("status") == "INGESTED")
    .orderBy(F.col("created_at").desc())
    .select("ingestion_batch_id", "created_at", "source_count", "status")
    .first()
)

if latest_manifest_row is None:
    raise ValueError("No INGESTED batch found in bronze_ext_ingestion_batch_manifest.")

INGESTION_BATCH_ID = latest_manifest_row["ingestion_batch_id"]

print("Using INGESTION_BATCH_ID:", INGESTION_BATCH_ID)
print("Batch created_at:", latest_manifest_row["created_at"])
print("Source count:", latest_manifest_row["source_count"])

external_inventory_rows = []

for table_name in external_bronze_tables:
    full_table_name = f"{BRONZE_SCHEMA}.{table_name}"

    try:
        df = spark.table(full_table_name)

        if "ingestion_batch_id" in df.columns:
            row_count = df.filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID).count()
        else:
            row_count = df.count()

        status = "FOUND"

        external_inventory_rows.append({
            "table_name": table_name,
            "full_table_name": full_table_name,
            "row_count": row_count,
            "status": status
        })

        print(f"{table_name}: {row_count} rows")

    except Exception as e:
        external_inventory_rows.append({
            "table_name": table_name,
            "full_table_name": full_table_name,
            "row_count": 0,
            "status": "MISSING"
        })

        print(f"{table_name}: MISSING")


external_inventory_schema = T.StructType([
    T.StructField("table_name", T.StringType(), True),
    T.StructField("full_table_name", T.StringType(), True),
    T.StructField("row_count", T.LongType(), True),
    T.StructField("status", T.StringType(), True)
])

external_inventory_df = spark.createDataFrame(external_inventory_rows, schema=external_inventory_schema)

display(external_inventory_df.orderBy("table_name"))

missing_table_count = external_inventory_df.filter(F.col("status") == "MISSING").count()

print("Missing external Bronze tables:", missing_table_count)

if missing_table_count == 0:
    print("✅ Chunk 1 PASSED: All external Bronze tables are available.")
else:
    print("❌ Chunk 1 FAILED: Some external Bronze tables are missing.")

In [0]:
# Notebook 13, Chunk 2: Build normalized Silver external risk event DataFrames

def add_external_event_metadata(df):
    return (
        df
        .withColumn("ingestion_batch_id", F.lit(INGESTION_BATCH_ID))
        .withColumn("silver_transform_run_id", F.lit(TRANSFORM_RUN_ID))
        .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
        .withColumn("silver_created_by", F.lit(CREATED_BY))
        .withColumn("silver_notebook_name", F.lit(NOTEBOOK_NAME))
    )


# -----------------------------
# 1. GDELT news articles
# -----------------------------
gdelt_article_schema = T.StructType([
    T.StructField("articles", T.ArrayType(T.StructType([
        T.StructField("url", T.StringType(), True),
        T.StructField("title", T.StringType(), True),
        T.StructField("seendate", T.StringType(), True),
        T.StructField("domain", T.StringType(), True),
        T.StructField("language", T.StringType(), True),
        T.StructField("sourcecountry", T.StringType(), True),
        T.StructField("socialimage", T.StringType(), True)
    ])), True)
])

gdelt_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_gdelt_doc_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

gdelt_events_df = (
    gdelt_raw_df
    .withColumn("parsed_payload", F.from_json(F.col("raw_payload_json"), gdelt_article_schema))
    .withColumn("article", F.explode_outer(F.col("parsed_payload.articles")))
    .filter(F.col("article.url").isNotNull())
    .select(
        F.lit("gdelt").alias("source_name"),
        F.lit("news_disruption").alias("risk_category"),
        F.lit("supply_chain_news").alias("event_type"),
        F.to_timestamp(F.col("article.seendate"), "yyyyMMdd'T'HHmmss'Z'").alias("event_timestamp"),
        F.to_date(F.to_timestamp(F.col("article.seendate"), "yyyyMMdd'T'HHmmss'Z'")).alias("event_date"),
        F.col("article.title").alias("event_title"),
        F.col("article.title").alias("event_summary"),
        F.col("article.url").alias("source_url"),
        F.col("article.domain").alias("source_domain"),
        F.col("article.sourcecountry").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.col("article.language").alias("language"),
        F.lit("medium").alias("severity"),
        F.lit("article").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

gdelt_events_df = add_external_event_metadata(gdelt_events_df)


# -----------------------------
# 2. NWS weather alerts
# -----------------------------
nws_schema = T.StructType([
    T.StructField("features", T.ArrayType(T.StructType([
        T.StructField("id", T.StringType(), True),
        T.StructField("properties", T.StructType([
            T.StructField("event", T.StringType(), True),
            T.StructField("severity", T.StringType(), True),
            T.StructField("urgency", T.StringType(), True),
            T.StructField("certainty", T.StringType(), True),
            T.StructField("areaDesc", T.StringType(), True),
            T.StructField("headline", T.StringType(), True),
            T.StructField("description", T.StringType(), True),
            T.StructField("instruction", T.StringType(), True),
            T.StructField("sent", T.StringType(), True),
            T.StructField("effective", T.StringType(), True),
            T.StructField("onset", T.StringType(), True),
            T.StructField("expires", T.StringType(), True),
            T.StructField("senderName", T.StringType(), True)
        ]), True)
    ])), True)
])

nws_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_nws_alerts_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

nws_events_df = (
    nws_raw_df
    .withColumn("parsed_payload", F.from_json(F.col("raw_payload_json"), nws_schema))
    .withColumn("feature", F.explode_outer(F.col("parsed_payload.features")))
    .filter(F.col("feature.properties.event").isNotNull())
    .select(
        F.lit("nws").alias("source_name"),
        F.lit("weather_logistics").alias("risk_category"),
        F.col("feature.properties.event").alias("event_type"),
        F.to_timestamp(F.col("feature.properties.onset")).alias("event_timestamp"),
        F.to_date(F.to_timestamp(F.col("feature.properties.onset"))).alias("event_date"),
        F.col("feature.properties.headline").alias("event_title"),
        F.col("feature.properties.description").alias("event_summary"),
        F.col("feature.id").alias("source_url"),
        F.col("feature.properties.senderName").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.col("feature.properties.areaDesc").alias("event_region"),
        F.lit("en").alias("language"),
        F.lower(F.col("feature.properties.severity")).alias("severity"),
        F.lit("weather_alert").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

nws_events_df = add_external_event_metadata(nws_events_df)


# -----------------------------
# 3. CISA KEV vulnerabilities
# -----------------------------
cisa_schema = T.StructType([
    T.StructField("vulnerabilities", T.ArrayType(T.StructType([
        T.StructField("cveID", T.StringType(), True),
        T.StructField("vendorProject", T.StringType(), True),
        T.StructField("product", T.StringType(), True),
        T.StructField("vulnerabilityName", T.StringType(), True),
        T.StructField("dateAdded", T.StringType(), True),
        T.StructField("shortDescription", T.StringType(), True),
        T.StructField("requiredAction", T.StringType(), True),
        T.StructField("dueDate", T.StringType(), True),
        T.StructField("knownRansomwareCampaignUse", T.StringType(), True)
    ])), True)
])

cisa_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_cisa_kev_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

cisa_events_df = (
    cisa_raw_df
    .withColumn("parsed_payload", F.from_json(F.col("raw_payload_json"), cisa_schema))
    .withColumn("vulnerability", F.explode_outer(F.col("parsed_payload.vulnerabilities")))
    .filter(F.col("vulnerability.cveID").isNotNull())
    .select(
        F.lit("cisa").alias("source_name"),
        F.lit("cyber").alias("risk_category"),
        F.lit("known_exploited_vulnerability").alias("event_type"),
        F.to_timestamp(F.col("vulnerability.dateAdded")).alias("event_timestamp"),
        F.to_date(F.col("vulnerability.dateAdded")).alias("event_date"),
        F.concat_ws(" - ", F.col("vulnerability.cveID"), F.col("vulnerability.vulnerabilityName")).alias("event_title"),
        F.col("vulnerability.shortDescription").alias("event_summary"),
        F.lit("https://www.cisa.gov/known-exploited-vulnerabilities-catalog").alias("source_url"),
        F.lit("cisa.gov").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.lit("en").alias("language"),
        F.when(F.lower(F.col("vulnerability.knownRansomwareCampaignUse")) == "known", F.lit("high"))
         .otherwise(F.lit("medium")).alias("severity"),
        F.lit("cyber_catalog").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

cisa_events_df = add_external_event_metadata(cisa_events_df)


# -----------------------------
# 4. CPSC recalls
# -----------------------------
cpsc_schema = T.ArrayType(T.StructType([
    T.StructField("RecallNumber", T.StringType(), True),
    T.StructField("RecallDate", T.StringType(), True),
    T.StructField("Title", T.StringType(), True),
    T.StructField("Description", T.StringType(), True),
    T.StructField("URL", T.StringType(), True)
]))

cpsc_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_cpsc_recalls_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

cpsc_events_df = (
    cpsc_raw_df
    .withColumn("parsed_payload", F.from_json(F.col("raw_payload_json"), cpsc_schema))
    .withColumn("recall", F.explode_outer(F.col("parsed_payload")))
    .filter(F.col("recall.RecallNumber").isNotNull())
    .select(
        F.lit("cpsc").alias("source_name"),
        F.lit("quality_recall").alias("risk_category"),
        F.lit("product_recall").alias("event_type"),
        F.to_timestamp(F.col("recall.RecallDate")).alias("event_timestamp"),
        F.to_date(F.col("recall.RecallDate")).alias("event_date"),
        F.col("recall.Title").alias("event_title"),
        F.col("recall.Description").alias("event_summary"),
        F.col("recall.URL").alias("source_url"),
        F.lit("saferproducts.gov").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.lit("en").alias("language"),
        F.lit("medium").alias("severity"),
        F.lit("product_recall").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

cpsc_events_df = add_external_event_metadata(cpsc_events_df)


# -----------------------------
# 5. OFAC sanctions files, document-level v0
# -----------------------------
ofac_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_ofac_sanctions_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

ofac_events_df = (
    ofac_raw_df
    .select(
        F.lit("ofac").alias("source_name"),
        F.lit("sanctions_compliance").alias("risk_category"),
        F.col("endpoint_name").alias("event_type"),
        F.to_timestamp(F.col("fetched_at")).alias("event_timestamp"),
        F.to_date(F.to_timestamp(F.col("fetched_at"))).alias("event_date"),
        F.concat(F.lit("OFAC sanctions file snapshot: "), F.col("endpoint_name")).alias("event_title"),
        F.substring(F.col("raw_payload_json"), 1, 1000).alias("event_summary"),
        F.col("request_url").alias("source_url"),
        F.lit("treasury.gov").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.lit("en").alias("language"),
        F.lit("high").alias("severity"),
        F.lit("sanctions_snapshot").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

ofac_events_df = add_external_event_metadata(ofac_events_df)


# -----------------------------
# 6. SEC EDGAR company submissions, company-level v0
# -----------------------------
sec_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_sec_company_submissions_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

sec_events_df = (
    sec_raw_df
    .select(
        F.lit("sec_edgar").alias("source_name"),
        F.lit("financial_filing").alias("risk_category"),
        F.lit("company_filings_snapshot").alias("event_type"),
        F.to_timestamp(F.col("fetched_at")).alias("event_timestamp"),
        F.to_date(F.to_timestamp(F.col("fetched_at"))).alias("event_date"),
        F.concat(F.lit("SEC company submissions snapshot: "), F.col("company_name")).alias("event_title"),
        F.concat(
            F.lit("SEC filings metadata captured for "),
            F.col("company_name"),
            F.lit(" ("),
            F.col("ticker"),
            F.lit(", CIK "),
            F.col("cik"),
            F.lit(").")
        ).alias("event_summary"),
        F.col("request_url").alias("source_url"),
        F.lit("sec.gov").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.lit("en").alias("language"),
        F.lit("medium").alias("severity"),
        F.lit("filing_metadata").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

sec_events_df = add_external_event_metadata(sec_events_df)


# -----------------------------
# 7. EIA fuel price snapshot, document-level v0
# -----------------------------
eia_raw_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_eia_fuel_prices_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("response_ok") == True)
)

eia_events_df = (
    eia_raw_df
    .select(
        F.lit("eia").alias("source_name"),
        F.lit("fuel_logistics_cost").alias("risk_category"),
        F.lit("fuel_price_snapshot").alias("event_type"),
        F.to_timestamp(F.col("fetched_at")).alias("event_timestamp"),
        F.to_date(F.to_timestamp(F.col("fetched_at"))).alias("event_date"),
        F.lit("EIA fuel price data snapshot").alias("event_title"),
        F.lit("Fuel price data captured for logistics cost risk signals.").alias("event_summary"),
        F.col("request_url").alias("source_url"),
        F.lit("eia.gov").alias("source_domain"),
        F.lit("US").alias("event_country"),
        F.lit(None).cast("string").alias("event_region"),
        F.lit("en").alias("language"),
        F.lit("medium").alias("severity"),
        F.lit("fuel_price_snapshot").alias("evidence_type"),
        F.col("raw_payload_json").alias("raw_payload_json"),
        F.col("payload_hash").alias("source_payload_hash"),
        F.col("api_run_id").alias("api_run_id")
    )
)

eia_events_df = add_external_event_metadata(eia_events_df)


silver_external_risk_events_df = (
    gdelt_events_df
    .unionByName(nws_events_df)
    .unionByName(cisa_events_df)
    .unionByName(cpsc_events_df)
    .unionByName(ofac_events_df)
    .unionByName(sec_events_df)
    .unionByName(eia_events_df)
    .withColumn(
        "external_event_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("source_name"),
                F.col("risk_category"),
                F.col("event_type"),
                F.coalesce(F.col("event_title"), F.lit("")),
                F.coalesce(F.col("source_url"), F.lit("")),
                F.coalesce(F.col("event_date").cast("string"), F.lit(""))
            ),
            256
        )
    )
)

external_event_count = silver_external_risk_events_df.count()

print("Silver external risk event count:", external_event_count)

display(
    silver_external_risk_events_df
    .groupBy("source_name", "risk_category")
    .agg(F.count("*").alias("event_count"))
    .orderBy("source_name", "risk_category")
)

display(
    silver_external_risk_events_df
    .select(
        "external_event_id",
        "source_name",
        "risk_category",
        "event_type",
        "event_date",
        "event_title",
        "severity",
        "source_domain",
        "event_country",
        "event_region"
    )
    .limit(30)
)

print("✅ Chunk 2 complete: Normalized Silver external risk events dataframe created.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 13 — Cell 3 SERVERLESS SAFE
# Clean, dedupe, and lightweight validate external risk events
# ─────────────────────────────────────────────────────────────

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone
import uuid

assert "silver_external_risk_events_df" in globals(), (
    "silver_external_risk_events_df is missing. "
    "Run the source parsing/build cell first."
)

VALIDATION_RUN_ID = f"silver_external_validation_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}"

allowed_severities = [
    "low",
    "medium",
    "high",
    "severe",
    "extreme",
    "minor",
    "moderate",
    "unknown"
]

input_event_count = silver_external_risk_events_df.count()
print("Input external event rows:", input_event_count)

# Normalize required fields
silver_external_risk_events_df = (
    silver_external_risk_events_df
    .withColumn("source_name", F.lower(F.trim(F.col("source_name"))))
    .withColumn("risk_category", F.lower(F.trim(F.col("risk_category"))))
    .withColumn(
        "event_type",
        F.coalesce(
            F.col("event_type"),
            F.concat(F.coalesce(F.col("risk_category"), F.lit("unknown")), F.lit("_event"))
        )
    )
    .withColumn(
        "severity",
        F.when(
            F.lower(F.trim(F.col("severity"))).isin(allowed_severities),
            F.lower(F.trim(F.col("severity")))
        )
        .when(F.lower(F.trim(F.col("severity"))).isin("warning", "watch", "advisory"), F.lit("medium"))
        .when(F.lower(F.trim(F.col("severity"))).isin("critical"), F.lit("high"))
        .otherwise(F.lit("unknown"))
    )
    .withColumn("evidence_type", F.coalesce(F.col("evidence_type"), F.lit("external_api_payload")))
    .withColumn(
        "event_title",
        F.coalesce(
            F.col("event_title"),
            F.col("event_summary"),
            F.concat(
                F.lit("External risk event from "),
                F.coalesce(F.col("source_name"), F.lit("unknown_source"))
            )
        )
    )
    .withColumn("event_summary", F.coalesce(F.col("event_summary"), F.col("event_title")))
)

# Create external_event_id only if missing
if "external_event_id" not in silver_external_risk_events_df.columns:
    silver_external_risk_events_df = (
        silver_external_risk_events_df
        .withColumn(
            "external_event_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("source_name"), F.lit("unknown_source")),
                    F.coalesce(F.col("risk_category"), F.lit("unknown_category")),
                    F.coalesce(F.col("event_type"), F.lit("unknown_event_type")),
                    F.coalesce(F.col("event_title"), F.lit("unknown_title")),
                    F.coalesce(F.col("source_url"), F.lit("")),
                    F.coalesce(F.col("event_date").cast("string"), F.lit("")),
                    F.coalesce(F.col("source_payload_hash"), F.lit("")),
                    F.coalesce(F.col("api_run_id"), F.lit(""))
                ),
                256
            )
        )
    )

# Deterministic dedupe
event_dedupe_window = (
    Window
    .partitionBy("external_event_id")
    .orderBy(
        F.col("event_timestamp").desc_nulls_last(),
        F.col("event_date").desc_nulls_last(),
        F.col("source_name").asc_nulls_last(),
        F.col("source_payload_hash").asc_nulls_last(),
        F.col("api_run_id").asc_nulls_last()
    )
)

silver_external_risk_events_df = (
    silver_external_risk_events_df
    .withColumn("event_dedupe_rank", F.row_number().over(event_dedupe_window))
    .filter(F.col("event_dedupe_rank") == 1)
    .drop("event_dedupe_rank")
)

deduped_event_count = silver_external_risk_events_df.count()
print("Rows after dedupe:", deduped_event_count)

# Serverless-safe lightweight validation.
# Avoid repeated countDistinct / null-scan aggregations here.
validation_rows = [
    {
        "validation_check": "external_event_count_positive",
        "expected_value": "> 0",
        "actual_value": str(deduped_event_count),
        "status": "PASS" if deduped_event_count > 0 else "FAIL",
        "issue_detail": None if deduped_event_count > 0 else "No external events were created."
    },
    {
        "validation_check": "dedupe_completed",
        "expected_value": "Dedupe step completed by external_event_id",
        "actual_value": f"input={input_event_count}, deduped={deduped_event_count}",
        "status": "PASS",
        "issue_detail": None
    },
    {
        "validation_check": "raw_payload_column_available",
        "expected_value": "raw_payload_json column exists before split",
        "actual_value": str("raw_payload_json" in silver_external_risk_events_df.columns),
        "status": "PASS" if "raw_payload_json" in silver_external_risk_events_df.columns else "FAIL",
        "issue_detail": None if "raw_payload_json" in silver_external_risk_events_df.columns else "Missing raw_payload_json before evidence split."
    },
    {
        "validation_check": "required_join_columns_available",
        "expected_value": "api_run_id and source_payload_hash exist",
        "actual_value": str(
            ("api_run_id" in silver_external_risk_events_df.columns)
            and ("source_payload_hash" in silver_external_risk_events_df.columns)
        ),
        "status": "PASS" if (
            ("api_run_id" in silver_external_risk_events_df.columns)
            and ("source_payload_hash" in silver_external_risk_events_df.columns)
        ) else "FAIL",
        "issue_detail": None
    }
]

validation_schema = T.StructType([
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True),
])

external_event_validation_df = spark.createDataFrame(validation_rows, schema=validation_schema)

display(external_event_validation_df)

# Compute fail_count without another Spark count.
fail_count = sum(1 for row in validation_rows if row["status"] == "FAIL")
warn_count = sum(1 for row in validation_rows if row["status"] == "WARN")

print("Silver external event validation failures:", fail_count)
print("Silver external event validation warnings:", warn_count)

if fail_count != 0:
    raise ValueError("Silver external risk event validation failed. Fix issues before writing.")

print("✅ Cell 3 PASSED.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 13 — Cell 4 SAFE FINAL
# Split slim event rows from lightweight evidence documents
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F

assert "silver_external_risk_events_df" in globals(), "Run Cell 3 first."
assert "fail_count" in globals(), "Run Cell 3 first."
assert fail_count == 0, "Validation has failures. Do not split/write."

silver_external_risk_events_slim_df = silver_external_risk_events_df.drop("raw_payload_json")

evidence_group_cols = [
    "source_name",
    "api_run_id",
    "source_payload_hash",
    "raw_payload_json",
    "ingestion_batch_id",
    "silver_transform_run_id",
    "silver_created_at",
    "silver_created_by",
    "silver_notebook_name"
]

missing_cols = [
    col_name
    for col_name in evidence_group_cols
    if col_name not in silver_external_risk_events_df.columns
]

if missing_cols:
    raise ValueError(f"Missing columns required for evidence document split: {missing_cols}")

silver_external_evidence_documents_df = (
    silver_external_risk_events_df
    .select(*evidence_group_cols)
    .dropDuplicates(["source_name", "api_run_id", "source_payload_hash"])
    .withColumnRenamed("source_payload_hash", "payload_hash")
    .withColumn("raw_payload_size_chars", F.length(F.col("raw_payload_json")))
    .withColumn("raw_payload_preview", F.substring(F.col("raw_payload_json"), 1, 5000))
    .drop("raw_payload_json")
    .withColumn("evidence_storage_mode", F.lit("bronze_payload_pointer_with_preview"))
    .withColumn(
        "evidence_document_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("source_name"), F.lit("unknown_source")),
                F.coalesce(F.col("api_run_id"), F.lit("unknown_api_run")),
                F.coalesce(F.col("payload_hash"), F.lit("unknown_payload_hash"))
            ),
            256
        )
    )
    .select(
        "evidence_document_id",
        "source_name",
        "api_run_id",
        "payload_hash",
        "raw_payload_preview",
        "raw_payload_size_chars",
        "evidence_storage_mode",
        "ingestion_batch_id",
        "silver_transform_run_id",
        "silver_created_at",
        "silver_created_by",
        "silver_notebook_name"
    )
)

event_count = silver_external_risk_events_slim_df.count()
doc_count = silver_external_evidence_documents_df.count()

missing_key_count = (
    silver_external_risk_events_slim_df
    .select("api_run_id", "source_payload_hash")
    .dropDuplicates()
    .alias("e")
    .join(
        silver_external_evidence_documents_df
        .select(
            F.col("api_run_id"),
            F.col("payload_hash").alias("source_payload_hash")
        )
        .dropDuplicates()
        .alias("d"),
        ["api_run_id", "source_payload_hash"],
        "left_anti"
    )
    .count()
)

print("Slim external event rows:", event_count)
print("Lightweight evidence document rows:", doc_count)
print("raw_payload_json in slim events:", "raw_payload_json" in silver_external_risk_events_slim_df.columns)
print("raw_payload_json in evidence docs:", "raw_payload_json" in silver_external_evidence_documents_df.columns)
print("raw_payload_preview in evidence docs:", "raw_payload_preview" in silver_external_evidence_documents_df.columns)
print("Event evidence keys missing documents:", missing_key_count)

if event_count <= 0:
    raise ValueError("Slim external event DataFrame is empty.")

if doc_count <= 0:
    raise ValueError("Evidence document DataFrame is empty.")

if "raw_payload_json" in silver_external_risk_events_slim_df.columns:
    raise ValueError("Slim event table still contains raw_payload_json.")

if "raw_payload_json" in silver_external_evidence_documents_df.columns:
    raise ValueError("Evidence docs should not store full raw_payload_json in Silver.")

if missing_key_count != 0:
    raise ValueError("Some event source payload keys do not have evidence documents.")

print("✅ Cell 4 SAFE FINAL PASSED.")

In [0]:
from pyspark.sql import functions as F

SILVER_SCHEMA = globals().get("SILVER_SCHEMA", "supplysage_silver")
TARGET_EVENTS_TABLE = f"{SILVER_SCHEMA}.silver_external_risk_events"

assert "silver_external_risk_events_slim_df" in globals(), "Run the split cell (Cell 4) first."
assert "fail_count" in globals() and fail_count == 0, "Validation has failures. Do not write."

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

(
    silver_external_risk_events_slim_df
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_EVENTS_TABLE)
)

events_written = spark.table(TARGET_EVENTS_TABLE).count()
print(f"✅ Wrote {events_written} rows to {TARGET_EVENTS_TABLE}")
if events_written <= 0:
    raise ValueError(f"{TARGET_EVENTS_TABLE} is empty.")
print("✅ Cell 5 PASSED: events table persisted. Run Cell 5D to build evidence docs.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 13 — Cell 5D (REPLACEMENT, robust)
# Recreate lightweight evidence docs from the WRITTEN events table.
#
# This version is safe to run after the new Cell 5 (write) above.
# It no longer assumes the events table pre-exists from a prior manual run:
# Cell 5 guarantees it. If the table is somehow still missing, this raises a
# clear, actionable error instead of a raw TABLE_OR_VIEW_NOT_FOUND.
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

SILVER_SCHEMA = globals().get("SILVER_SCHEMA", "supplysage_silver")

TARGET_EVENTS_TABLE = f"{SILVER_SCHEMA}.silver_external_risk_events"
TARGET_EVIDENCE_DOCUMENTS_TABLE = f"{SILVER_SCHEMA}.silver_external_evidence_documents"

print("Using written events table:", TARGET_EVENTS_TABLE)
print("Recreating evidence docs table:", TARGET_EVIDENCE_DOCUMENTS_TABLE)

# Make sure events table exists and has rows. Fail loudly with guidance.
try:
    written_events_df = spark.table(TARGET_EVENTS_TABLE)
    written_event_count = written_events_df.count()
except AnalysisException as e:
    raise ValueError(
        f"{TARGET_EVENTS_TABLE} does not exist. "
        f"Run the write cell (new Cell 5) before this cell. "
        f"Underlying error: {e}"
    )

print("Written external event rows:", written_event_count)

if written_event_count <= 0:
    raise ValueError("Written external events table is empty.")

# The events table must carry source_payload_hash for the evidence-doc grouping.
if "source_payload_hash" not in written_events_df.columns:
    raise ValueError(
        "source_payload_hash column is missing from the written events table; "
        "cannot rebuild evidence documents."
    )

# Drop old evidence table if it exists
spark.sql(f"DROP TABLE IF EXISTS {TARGET_EVIDENCE_DOCUMENTS_TABLE}")
print("✅ Dropped old evidence docs table if it existed.")

# Create lightweight evidence document table from already-written event rows.
spark.sql(f"""
CREATE TABLE {TARGET_EVIDENCE_DOCUMENTS_TABLE}
USING DELTA
AS
SELECT
  sha2(
    concat_ws(
      '||',
      coalesce(source_name, 'unknown_source'),
      coalesce(api_run_id, 'unknown_api_run'),
      coalesce(source_payload_hash, 'unknown_payload_hash')
    ),
    256
  ) AS evidence_document_id,
  source_name,
  api_run_id,
  source_payload_hash AS payload_hash,
  CAST(NULL AS STRING) AS raw_payload_preview,
  CAST(NULL AS INT) AS raw_payload_size_chars,
  'bronze_payload_pointer_only' AS evidence_storage_mode,
  max(ingestion_batch_id) AS ingestion_batch_id,
  max(silver_transform_run_id) AS silver_transform_run_id,
  max(silver_created_at) AS silver_created_at,
  max(silver_created_by) AS silver_created_by,
  max(silver_notebook_name) AS silver_notebook_name
FROM {TARGET_EVENTS_TABLE}
GROUP BY
  source_name,
  api_run_id,
  source_payload_hash
""")

print("✅ Created lightweight evidence docs table:", TARGET_EVIDENCE_DOCUMENTS_TABLE)

# Minimal readback validation
written_docs_df = spark.table(TARGET_EVIDENCE_DOCUMENTS_TABLE)
written_doc_count = written_docs_df.count()

missing_key_count = (
    written_events_df
    .select("api_run_id", "source_payload_hash")
    .dropDuplicates()
    .alias("e")
    .join(
        written_docs_df
        .select(
            F.col("api_run_id"),
            F.col("payload_hash").alias("source_payload_hash")
        )
        .dropDuplicates()
        .alias("d"),
        ["api_run_id", "source_payload_hash"],
        "left_anti"
    )
    .count()
)

print("\nReadback validation:")
print("Written external event rows:", written_event_count)
print("Written evidence document rows:", written_doc_count)
print("raw_payload_json in events:", "raw_payload_json" in written_events_df.columns)
print("raw_payload_json in docs:", "raw_payload_json" in written_docs_df.columns)
print("raw_payload_preview in docs:", "raw_payload_preview" in written_docs_df.columns)
print("Event evidence keys missing documents:", missing_key_count)

if written_doc_count <= 0:
    raise ValueError("Written evidence documents table is empty.")

if "raw_payload_json" in written_events_df.columns:
    raise ValueError("Written events table should not contain raw_payload_json.")

if "raw_payload_json" in written_docs_df.columns:
    raise ValueError("Written evidence docs should not contain raw_payload_json.")

if missing_key_count != 0:
    raise ValueError("Some written event evidence keys do not join to evidence documents.")

print("\n✅ Notebook 13 PASSED: Silver external risk events and lightweight evidence pointers are written and validated.")
